# Databricks Environment Setup (00)
Idempotent setup for the learning environment. Safe to re-run. Nothing here grants permissions to you; if something fails, the cell prints the exact SQL an administrator must run.

**Creates:** catalog `databricks_learning`; schemas `bronze, silver, gold, sandbox, advanced`; volume `sandbox.learning_volume`; sample tables `customers, products, orders, payments, events` in `bronze`.

**Contents:** 1 Config, 2 Environment detection, 3 Unity Catalog check, 4 Catalog, 5 Schemas, 6 Volume, 7 Sample data, 8 Permissions inspection, 9 Secrets, 10 Items outside notebooks (warehouses, service principals, external locations), 11 Validation summary, 12 Cleanup.

**Compute:** serverless or any UC-enabled compute (standard/dedicated access mode). Runtime 13.3 LTS or later.

## 1. Config and helper

In [ ]:
import sys, random, datetime as dt
from pyspark.sql import functions as F, types as T

CATALOG = "databricks_learning"
SCHEMAS = ["bronze", "silver", "gold", "sandbox", "advanced"]
VOLUME_SCHEMA, VOLUME = "sandbox", "learning_volume"
VOLUME_PATH = f"/Volumes/{CATALOG}/{VOLUME_SCHEMA}/{VOLUME}"
RESULTS = []          # (step, status, detail)

def step(name, fn, skip_reason=None):
    """Run fn, record PASSED / FAILED / SKIPPED, never raise."""
    if skip_reason:
        RESULTS.append((name, "SKIPPED", skip_reason)); print(f"[SKIPPED] {name}: {skip_reason}"); return None
    try:
        out = fn()
        RESULTS.append((name, "PASSED", "")); print(f"[PASSED]  {name}")
        return out
    except Exception as e:
        msg = str(e).strip().splitlines()[0][:300] if str(e).strip() else type(e).__name__
        RESULTS.append((name, "FAILED", msg)); print(f"[FAILED]  {name}: {msg}")
        return None

def ok(name):
    return any(r[0] == name and r[1] == "PASSED" for r in RESULTS)

## 2. Environment detection
Some Spark confs do not exist on serverless; missing values show as `n/a`.

In [ ]:
def conf(key):
    try:
        return spark.conf.get(key)
    except Exception:
        return None

def detect_cloud():
    c = conf("spark.databricks.cloudProvider") or conf("spark.databricks.clusterUsageTags.cloudProvider")
    if c:
        return c
    url = (conf("spark.databricks.workspaceUrl") or "").lower()
    if "azuredatabricks.net" in url: return "Azure"
    if "gcp.databricks.com" in url: return "GCP"
    if "databricks.com" in url: return "AWS"
    return "unknown"

info = {
    "spark_version": spark.version,
    "python_version": sys.version.split()[0],
    "databricks_runtime": conf("spark.databricks.clusterUsageTags.sparkVersion") or "n/a (likely serverless)",
    "cloud": detect_cloud(),
    "workspace_url": conf("spark.databricks.workspaceUrl") or "n/a",
    "current_user": spark.sql("SELECT current_user()").first()[0],
}
for k, v in info.items():
    print(f"{k:20} {v}")
step("spark session", lambda: spark.sql("SELECT 1").collect())

## 3. Unity Catalog / metastore check

In [ ]:
def uc_check():
    ms = spark.sql("SELECT current_metastore()").first()[0]
    print("metastore:", ms)
    print("catalogs :", [r[0] for r in spark.sql("SHOW CATALOGS").collect()])
step("unity catalog available", uc_check)

## 4. Catalog
Requires `CREATE CATALOG` on the metastore. If the metastore has no default storage root, add `MANAGED LOCATION 's3://... | abfss://... | gs://...'` (an external location you can use).

In [ ]:
def create_catalog():
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} COMMENT 'Databricks learning environment'")
step("create catalog", create_catalog)

if not ok("create catalog"):
    me = info["current_user"]
    print(f"""
ADMIN ACTION REQUIRED (metastore admin runs this):
  GRANT CREATE CATALOG ON METASTORE TO `{me}`;
If the metastore has no default storage, create the catalog with a location:
  CREATE CATALOG {CATALOG} MANAGED LOCATION '<cloud-path-you-have-an-external-location-for>';
Alternative without a new catalog: set CATALOG above to one you already own.
""")
else:
    spark.sql(f"USE CATALOG {CATALOG}")
    print(spark.sql(f"DESCRIBE CATALOG EXTENDED {CATALOG}").toPandas().to_string())

## 5. Schemas

In [ ]:
def create_schemas():
    for s in SCHEMAS:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{s} COMMENT 'learning: {s}'")
    have = [r[0] for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]
    missing = [s for s in SCHEMAS if s not in have]
    assert not missing, f"missing schemas: {missing}"
step("create schemas", create_schemas, None if ok("create catalog") else "catalog unavailable")

## 6. Managed volume
Volumes hold files (CSV, JSON, etc.) under `/Volumes/<catalog>/<schema>/<volume>/`. Needs `CREATE VOLUME` on the schema.

In [ ]:
def create_volume():
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{VOLUME_SCHEMA}.{VOLUME} COMMENT 'learning files'")
    spark.sql(f"DESCRIBE VOLUME {CATALOG}.{VOLUME_SCHEMA}.{VOLUME}").show(truncate=False)
step("create volume", create_volume, None if ok("create schemas") else "schemas unavailable")

## 7. Sample data (generated, no uploads)

In [ ]:
random.seed(42)
N_CUST, N_PROD, N_ORD = 200, 30, 1000
cities = ["Hyderabad", "Pune", "Mumbai", "Delhi", "Bengaluru", "Chennai"]
cats = ["electronics", "books", "home", "fashion", "toys"]
base = dt.datetime(2026, 1, 1)

customers = [(i, f"Customer {i}", f"user{i}@example.com", random.choice(cities),
              (base + dt.timedelta(days=random.randint(0, 300))).date()) for i in range(1, N_CUST + 1)]
products = [(i, f"Product {i}", random.choice(cats), round(random.uniform(5, 500), 2)) for i in range(1, N_PROD + 1)]
price = {p[0]: p[3] for p in products}
orders, payments = [], []
for oid in range(1, N_ORD + 1):
    pid, qty = random.randint(1, N_PROD), random.randint(1, 5)
    ts = base + dt.timedelta(minutes=random.randint(0, 60 * 24 * 300))
    amount = round(price[pid] * qty, 2)
    status = random.choices(["PAID", "PENDING", "CANCELLED"], [0.8, 0.1, 0.1])[0]
    orders.append((oid, random.randint(1, N_CUST), pid, qty, amount, status, ts))
    if status == "PAID":
        payments.append((oid, oid, random.choice(["card", "upi", "netbanking"]), amount, ts + dt.timedelta(minutes=2)))
events = [(i, random.randint(1, N_CUST), random.choice(["view", "click", "add_to_cart", "purchase"]),
           base + dt.timedelta(seconds=random.randint(0, 86400 * 300))) for i in range(1, 2001)]

specs = {
    "customers": (customers, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    "products":  (products,  "product_id INT, name STRING, category STRING, price DOUBLE"),
    "orders":    (orders,    "order_id INT, customer_id INT, product_id INT, quantity INT, amount DOUBLE, status STRING, order_ts TIMESTAMP"),
    "payments":  (payments,  "payment_id INT, order_id INT, method STRING, amount DOUBLE, paid_ts TIMESTAMP"),
    "events":    (events,    "event_id INT, customer_id INT, event_type STRING, event_ts TIMESTAMP"),
}

def load_samples():
    for name, (rows, ddl) in specs.items():
        df = spark.createDataFrame(rows, schema=ddl)
        (df.write.mode("overwrite").option("overwriteSchema", "true")
           .saveAsTable(f"{CATALOG}.bronze.{name}"))
        print(f"{CATALOG}.bronze.{name}: {spark.table(f'{CATALOG}.bronze.{name}').count()} rows")
step("load sample tables", load_samples, None if ok("create schemas") else "schemas unavailable")

def write_volume_files():
    import os
    os.makedirs(f"{VOLUME_PATH}/raw", exist_ok=True)
    spark.table(f"{CATALOG}.bronze.orders").limit(100).toPandas().to_csv(f"{VOLUME_PATH}/raw/orders_sample.csv", index=False)
    spark.table(f"{CATALOG}.bronze.customers").limit(50).toPandas().to_json(f"{VOLUME_PATH}/raw/customers_sample.json", orient="records", lines=True)
    print([f.name for f in dbutils.fs.ls(f"{VOLUME_PATH}/raw")])
step("write volume files", write_volume_files, None if ok("create volume") and ok("load sample tables") else "volume or tables unavailable")

## 8. Permissions: inspect and (as an owner) manage
You can only grant on objects you own or have `MANAGE` on. These cells inspect; the GRANT examples are commented out and use a placeholder group you must replace.

In [ ]:
def show_grants():
    spark.sql(f"SHOW GRANTS ON CATALOG {CATALOG}").show(truncate=False)
    spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG}.bronze").show(truncate=False)
step("inspect grants", show_grants, None if ok("create schemas") else "catalog unavailable")

# Owner/admin only. Replace `analysts` with a real group, then uncomment:
# spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `analysts`")
# spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOG}.bronze TO `analysts`")
# spark.sql(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{VOLUME_SCHEMA}.{VOLUME} TO `analysts`")
# spark.sql(f"REVOKE SELECT ON SCHEMA {CATALOG}.bronze FROM `analysts`")

## 9. Secrets
Scopes are created with the CLI or REST API only (not from a notebook). This cell only lists what exists.

In [ ]:
def list_scopes():
    print([s.name for s in dbutils.secrets.listScopes()])
step("list secret scopes", list_scopes)

## 10. Resources that cannot be created from a notebook
Run in a terminal with the Databricks CLI (authenticated via `databricks auth login`), or in the UI.

| Resource | Where | Command / step |
|---|---|---|
| Secret scope | CLI | `databricks secrets create-scope learning` then `databricks secrets put-secret learning my_key` |
| SQL warehouse | UI / CLI | `databricks warehouses create --json @wh.json`, list with `databricks warehouses list` |
| Service principal | Account console / CLI | `databricks service-principals create --display-name learning-sp` (account-level admin rights may be needed) |
| Storage credential | Catalog Explorer / SQL | Create in UI (cloud IAM role, managed identity or GCP service account); needs `CREATE STORAGE CREDENTIAL` |
| External location | SQL | `CREATE EXTERNAL LOCATION IF NOT EXISTS learning_loc URL '<cloud path>' WITH (STORAGE CREDENTIAL <cred>)`, needs `CREATE EXTERNAL LOCATION` and a credential |
| External table | SQL | `CREATE TABLE databricks_learning.advanced.ext_orders LOCATION '<path under an external location>' AS SELECT ...` |

These are intentionally not executed: they depend on your cloud, your permissions and paths that I cannot know.

## 11. Validation summary

In [ ]:
def validate():
    checks = {}
    checks["catalog exists"] = CATALOG in [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    if checks["catalog exists"]:
        have = [r[0] for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]
        for s in SCHEMAS:
            checks[f"schema {s}"] = s in have
        vols = [r[2] if len(r) > 2 else r[0] for r in spark.sql(f"SHOW VOLUMES IN {CATALOG}.{VOLUME_SCHEMA}").collect()]
        checks["volume learning_volume"] = VOLUME in vols
        tbls = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {CATALOG}.bronze").collect()]
        for t in specs:
            checks[f"table bronze.{t}"] = t in tbls
    return checks

checks = validate()
for k, v in checks.items():
    print(("OK   " if v else "MISS ") + k)

print("\nStep summary")
for name, status, detail in RESULTS:
    print(f"{status:8} {name}" + (f"  -> {detail}" if detail else ""))

## 12. Cleanup (destructive, learning catalog only)
Set `CONFIRM_CLEANUP = True` to drop everything this notebook created. The guard refuses any catalog other than `databricks_learning`.

In [ ]:
CONFIRM_CLEANUP = False

if CONFIRM_CLEANUP and CATALOG == "databricks_learning":
    spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")
    print("dropped", CATALOG)
else:
    print("Cleanup skipped (CONFIRM_CLEANUP is False).")